# Voice MTL heads — emotion + affect (CCC) + crisis (recall floor)

Frozen **emotion2vec_base** / **WavLM-Large** -> a shared **SUPERB trunk** -> **three heads**: emotion (8-way CE), affect regression (valence+arousal, **CCC**), and a **crisis** head (BCE) under a **hard recall floor**, balanced by **Kendall uncertainty weighting**. RAVDESS has no continuous/crisis labels, so the two new heads learn **proxy** targets (Russell circumplex V/A; high-distress emotion set). Validates the multi-head + recall-floor mechanics; real numbers need MSP-Podcast / DAIC. Protocol + rationale: `docs/tasks/voice-mtl-heads.md`.

## 0. Install pinned audio stack  (run once)

In [ ]:
# Cell 0 — pinned audio stack (P100 = sm_60; Kaggle default torch drops it -> crash).
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
# torchvision pinned to match torch 2.5.1 — else transformers' lazy torchvision import
# fails with "operator torchvision::nms does not exist" and WavLM won't load.
get_ipython().system('pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121')
get_ipython().system('pip install -q transformers==4.48.2 datasets==3.2.0 librosa soundfile scikit-learn scipy')
# emotion2vec route (primary/repro backbone). Heavy dep tree; usage is guarded so a
# failed install/import only skips the emotion2vec arm — WavLM still produces a result.
get_ipython().system('pip install -q funasr modelscope || echo "funasr install failed -> emotion2vec arm will be skipped"')
print("install cell done")


## 1. Imports & config

In [ ]:
# Cell 1 — imports & config.
# VOICE MTL HEADS (thesis extension, NOT a paper reproduction).
#   Frozen encoder -> a SHARED SUPERB trunk -> THREE heads:
#     - emotion   : Linear(256, 8)  cross-entropy        (WA/UA/WF1)
#     - affect    : Linear(256, 2)  CCC on valence+arousal
#     - crisis    : Linear(256, 1)  BCE under a HARD recall floor
#   balanced by Kendall et al. (CVPR 2018) homoscedastic uncertainty weighting.
# RAVDESS has no continuous/crisis labels, so the two NEW heads learn PROXY targets:
#   valence/arousal from a fixed Russell (1980) circumplex map of the 8 emotions, and
#   crisis = the high-distress emotion set. This validates the multi-head + recall-floor
#   MECHANICS on the reproduction's frozen features; real CCC/crisis numbers need the
#   continuous/clinical sets (MSP-Podcast A/V/D, DAIC). See docs/tasks/voice-mtl-heads.md.
import os, json, random, warnings, tempfile
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score
from sklearn.model_selection import train_test_split

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ART = "/kaggle/working"
SR = 16000
MAX_SEC = 5.0                          # RAVDESS clips ~3-5s; 5s covers virtually all
MAX_SAMPLES = int(SR * MAX_SEC)
FRAME_HOP = 320                        # both encoders downsample 16kHz->50Hz (320x)
T_MAX = MAX_SAMPLES // FRAME_HOP        # 250 frames; frame-level feats (T x d)

N_FOLDS = 10                           # random 10-fold CV (same protocol as the repro)
SPLIT = (0.80, 0.10, 0.10)            # 80/10/10 train/val/test per fold
BASE_SEED = 42
PROBE_EPOCHS, PROBE_LR, HEAD_DIM, WD = 100, 1e-3, 256, 1e-4
RECALL_FLOOR = 0.90                     # hard crisis-recall floor (thesis novelty mechanic)

# RAVDESS 8 emotions, canonical id order (filename code 01..08 -> 0..7).
EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
EMO2ID = {e: i for i, e in enumerate(EMOTIONS)}
N_EMO = len(EMOTIONS)

# PROXY targets for the two new heads (RAVDESS has no continuous/crisis labels):
#   Russell (1980) circumplex (valence, arousal) in [-1, 1] per emotion.
VALENCE_AROUSAL = {
    "neutral":  (0.0,  0.0), "calm":    (0.4, -0.6), "happy":   (0.8,  0.5),
    "sad":     (-0.6, -0.4), "angry":  (-0.6,  0.8), "fearful": (-0.6,  0.6),
    "disgust": (-0.7,  0.2), "surprised": (0.3,  0.7),
}
#   crisis / high-distress proxy -> safety head positive class.
CRISIS = {"angry", "fearful", "sad", "disgust"}
VA = np.array([VALENCE_AROUSAL[e] for e in EMOTIONS], dtype=np.float32)        # (8, 2)
SAFE = np.array([1.0 if e in CRISIS else 0.0 for e in EMOTIONS], dtype=np.float32)  # (8,)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"emotions={EMOTIONS}\nfolds={N_FOLDS} split={SPLIT} base_seed={BASE_SEED} "
      f"probe(ep={PROBE_EPOCHS},lr={PROBE_LR},hid={HEAD_DIM}) | recall_floor={RECALL_FLOOR} "
      f"| crisis={sorted(CRISIS)}")


## 2. RAVDESS data  (all 1440 clips, resample 48k->16k, no fixed split)

In [ ]:
# Cell 2 — RAVDESS load (all 1440 speech clips), resample 48k->16k. No fixed split:
# folds are drawn randomly in the probe cell (paper protocol).
import torchaudio
from datasets import load_dataset
import soundfile as sf

ds = load_dataset("narad/ravdess", split="train", trust_remote_code=True)
print("RAVDESS:", ds)
INT2STR = ds.features["labels"].int2str

def fix_len(wav):
    # pad/truncate to 5s; also return the true (pre-pad) valid frame count for masking.
    n = min(len(wav), MAX_SAMPLES)
    flen = int(np.clip(round(n / FRAME_HOP), 1, T_MAX))
    if len(wav) >= MAX_SAMPLES:
        return wav[:MAX_SAMPLES], flen
    return np.pad(wav, (0, MAX_SAMPLES - len(wav))), flen

def to_records(ds):
    recs = []
    for ex in ds:
        a = ex["audio"]
        wav = np.asarray(a["array"], dtype=np.float32)
        src_sr = a["sampling_rate"]
        if src_sr != SR:
            wav = torchaudio.functional.resample(torch.from_numpy(wav), src_sr, SR).numpy()
        wav, flen = fix_len(wav.astype(np.float32))
        emo = INT2STR(ex["labels"])
        recs.append({"wav": wav, "y": EMO2ID[emo], "flen": flen})
    return recs

recs = to_records(ds)
y_all = np.array([r["y"] for r in recs])
print(f"clips={len(recs)} | per-class counts={np.bincount(y_all, minlength=N_EMO).tolist()}")
assert len(recs) == 1440, f"expected 1440 RAVDESS speech clips, got {len(recs)}"

# stash one clip as the local FastAPI/test sample (16 kHz mono wav).
sample = recs[0]
sf.write(f"{ART}/sample_val.wav", sample["wav"], SR)
print(f"sample wav -> {ART}/sample_val.wav (emo={EMOTIONS[sample['y']]})")


## 3. Frozen feature extraction  (each encoder runs once over all clips)

In [ ]:
# Cell 3 — frozen FRAME-LEVEL feature extraction (faithful SUPERB recipe).
# Each encoder runs ONCE over all 1440 clips, returning (N, T_MAX, dim) frame features
# (padded/truncated to T_MAX); the probe pools AFTER the first Linear+ReLU using the
# true frame lengths (c2 `flen`) as a mask — matching emotion2vec/EmoBox SuperbBaseModel.
# emotion2vec = funasr granularity="frame" (T,768); WavLM-Large = last_hidden_state (T,1024).
from transformers import AutoFeatureExtractor, WavLMModel
import soundfile as sf

def pad_T(arr):
    # arr (T, dim) -> (T_MAX, dim) padded with zeros or truncated.
    if len(arr) >= T_MAX:
        return arr[:T_MAX]
    return np.pad(arr, ((0, T_MAX - len(arr)), (0, 0)))

def wavlm_extractor():
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
    model = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DEVICE).eval()
    @torch.no_grad()
    def extract(wavs, bs=16):
        out = []
        for i in range(0, len(wavs), bs):
            batch = [w for w in wavs[i:i + bs]]
            enc = fe(batch, sampling_rate=SR, return_tensors="pt", padding=True)
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                h = model(**enc).last_hidden_state            # (B, T, 1024)
            h = h.float().cpu().numpy()
            for j in range(h.shape[0]):
                out.append(pad_T(h[j]).astype(np.float16))    # keep time dim
        return np.stack(out)                                  # (N, T_MAX, 1024)
    return extract, 1024

def emotion2vec_extractor():
    from funasr import AutoModel as FunASR
    try:
        model = FunASR(model="iic/emotion2vec_base", hub="hf", disable_update=True)
    except Exception:
        from huggingface_hub import snapshot_download
        local = snapshot_download("emotion2vec/emotion2vec_base")
        model = FunASR(model=local, disable_update=True)
    def extract(wavs, bs=None):
        feats = []
        for w in wavs:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                sf.write(tf.name, w, SR); path = tf.name
            r = model.generate(path, granularity="frame", extract_embedding=True,
                               output_dir=None)
            f = np.asarray(r[0]["feats"], dtype=np.float32)    # (T, 768)
            os.remove(path)
            feats.append(pad_T(f).astype(np.float16))
        return np.stack(feats)                                 # (N, T_MAX, 768)
    return extract, 768

all_wavs = [r["wav"] for r in recs]
flen_all = np.array([r["flen"] for r in recs])
BACKBONES = {}   # name -> {"X": (N,T_MAX,dim) f16, "dim": int}
for name, builder in [("emotion2vec", emotion2vec_extractor), ("wavlm-large", wavlm_extractor)]:
    try:
        print(f"\n[{name}] building extractor...")
        extract, dim = builder()
        X = extract(all_wavs)
        BACKBONES[name] = {"X": X, "dim": dim}
        print(f"[{name}] frame features: {X.shape} ({X.dtype}) | flen mean={flen_all.mean():.0f}")
        del extract; torch.cuda.empty_cache()
    except Exception as e:
        print(f"[{name}] SKIPPED -> {type(e).__name__}: {e}")

assert BACKBONES, "no backbone produced features"
print("\nbackbones with features:", list(BACKBONES))


## 4. 3-head MTL probe  (random 10-fold CV; Kendall weighting; recall floor)

In [ ]:
# Cell 4 — 3-head MTL probe (shared SUPERB trunk) + Kendall uncertainty weighting,
# random 10-fold CV.
#   trunk = Linear(d,256) -> ReLU -> masked-mean-pool over valid frames  (= repro v2 trunk)
#   heads = emotion Linear(256,8) CE | affect Linear(256,2) CCC(valence,arousal) | crisis Linear(256,1) BCE
#   loss  = Kendall homoscedastic uncertainty weighting over the 3 task losses
# Per fold: train on 80%, select best epoch by VAL total loss, report on 10% test —
#   emotion WA/UA/WF1, valence/arousal CCC, and crisis recall/precision at a threshold
#   tuned ON VAL to guarantee recall >= RECALL_FLOOR (the hard-floor mechanic).
class MTLHead(nn.Module):
    def __init__(self, dim, hid=HEAD_DIM):
        super().__init__()
        self.pre = nn.Linear(dim, hid)        # shared SUPERB trunk
        self.emo = nn.Linear(hid, N_EMO)      # emotion (8-way)
        self.reg = nn.Linear(hid, 2)          # affect: valence, arousal
        self.safe = nn.Linear(hid, 1)         # crisis (binary)
    def forward(self, x, mask):               # x (B,T,dim), mask (B,T)
        h = F.relu(self.pre(x))
        m = mask.unsqueeze(-1).float()
        z = (h * m).sum(1) / m.sum(1).clamp(min=1.0)   # masked-mean utterance embedding
        return self.emo(z), self.reg(z), self.safe(z).squeeze(-1)

class UncertaintyWeighter(nn.Module):
    # Kendall, Gal & Cipolla (CVPR 2018) — homoscedastic uncertainty weighting over tasks.
    def __init__(self, n=3):
        super().__init__()
        self.log_var = nn.Parameter(torch.zeros(n))
    def forward(self, losses):
        return sum(0.5 * torch.exp(-self.log_var[i]) * L + 0.5 * self.log_var[i]
                   for i, L in enumerate(losses))

def ccc_loss(pred, tgt):
    pm, tm = pred.mean(), tgt.mean()
    pv, tv = pred.var(unbiased=False), tgt.var(unbiased=False)
    cov = ((pred - pm) * (tgt - tm)).mean()
    return 1 - 2 * cov / (pv + tv + (pm - tm) ** 2 + 1e-8)

def ccc_score(pred, tgt):
    pm, tm = pred.mean(), tgt.mean()
    cov = ((pred - pm) * (tgt - tm)).mean()
    return float(2 * cov / (pred.var() + tgt.var() + (pm - tm) ** 2 + 1e-8))

def threshold_for_recall(probs, labels, floor):
    # highest threshold (best precision) whose recall on these (val) examples is >= floor.
    pos = np.sort(probs[labels == 1])
    if len(pos) == 0:
        return 0.5
    k = min(int(np.floor((1 - floor) * len(pos))), len(pos) - 1)
    return float(pos[k])

def split_80_10_10(y, seed):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=SPLIT[1] + SPLIT[2], random_state=seed, stratify=y)
    rel = SPLIT[2] / (SPLIT[1] + SPLIT[2])
    va, te = train_test_split(tmp, test_size=rel, random_state=seed, stratify=y[tmp])
    return tr, va, te

def emo_metrics(y_true, y_pred):
    return {"WA": 100 * accuracy_score(y_true, y_pred),
            "UA": 100 * recall_score(y_true, y_pred, average="macro", zero_division=0),
            "WF1": 100 * f1_score(y_true, y_pred, average="weighted", zero_division=0)}

def train_one(Xg, mask_g, y, va_tgt, safe_tgt, dim, seed):
    tr, va, te = split_80_10_10(y, seed)
    set_seed(seed)
    yt = torch.tensor(y, dtype=torch.long, device=DEVICE)
    vt = torch.tensor(va_tgt, dtype=torch.float, device=DEVICE)     # (N,2)
    st = torch.tensor(safe_tgt, dtype=torch.float, device=DEVICE)   # (N,)
    head = MTLHead(dim).to(DEVICE)
    weighter = UncertaintyWeighter(3).to(DEVICE)
    opt = torch.optim.Adam(list(head.parameters()) + list(weighter.parameters()),
                           lr=PROBE_LR, weight_decay=WD)
    tr_t = torch.tensor(tr, device=DEVICE)
    best_val, best_state = 1e9, None
    for _ in range(PROBE_EPOCHS):
        head.train(); weighter.train(); opt.zero_grad()
        e, r, s = head(Xg[tr_t], mask_g[tr_t])
        losses = [F.cross_entropy(e, yt[tr_t]),
                  ccc_loss(r[:, 0], vt[tr_t, 0]) + ccc_loss(r[:, 1], vt[tr_t, 1]),
                  F.binary_cross_entropy_with_logits(s, st[tr_t])]
        loss = weighter(losses)
        loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            e, r, s = head(Xg[va], mask_g[va])
            vl = (F.cross_entropy(e, yt[va]) + ccc_loss(r[:, 0], vt[va, 0])
                  + ccc_loss(r[:, 1], vt[va, 1])
                  + F.binary_cross_entropy_with_logits(s, st[va])).item()
        if vl < best_val:
            best_val = vl
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
    head.load_state_dict(best_state); head.eval()
    with torch.no_grad():
        _, _, sv = head(Xg[va], mask_g[va])
        et, rt, st_ = head(Xg[te], mask_g[te])
    # hard crisis-recall floor: tune tau on val, apply to test.
    tau = threshold_for_recall(torch.sigmoid(sv).cpu().numpy(), safe_tgt[va], RECALL_FLOOR)
    pred = (torch.sigmoid(st_).cpu().numpy() >= tau).astype(int)
    m = emo_metrics(y[te], et.argmax(-1).cpu().numpy())
    m.update({"CCC_v": ccc_score(rt[:, 0].cpu().numpy(), va_tgt[te, 0]),
              "CCC_a": ccc_score(rt[:, 1].cpu().numpy(), va_tgt[te, 1]),
              "crisis_recall": float(recall_score(safe_tgt[te], pred, zero_division=0)),
              "crisis_prec": float(precision_score(safe_tgt[te], pred, zero_division=0)),
              "tau": tau})
    return m, head

# frame mask (N, T): True for valid frames; proxy targets from c1 maps.
mask_all = (torch.arange(T_MAX)[None, :] < torch.tensor(flen_all)[:, None]).to(DEVICE)
va_all = VA[y_all]        # (N, 2) valence/arousal
safe_all = SAFE[y_all]    # (N,)  crisis 0/1

results = {}     # name -> list of per-fold metric dicts
artifacts = {}   # name -> head from fold 0
for name, b in BACKBONES.items():
    Xg = torch.tensor(b["X"], dtype=torch.float, device=DEVICE)   # (N,T,dim) f16->f32 on GPU
    results[name] = []
    for fold in range(N_FOLDS):
        m, head = train_one(Xg, mask_all, y_all, va_all, safe_all, b["dim"], BASE_SEED + fold)
        results[name].append(m)
        if fold == 0:
            artifacts[name] = head
        print(f"[{name} f{fold}] WA={m['WA']:.1f} CCCv={m['CCC_v']:.2f} CCCa={m['CCC_a']:.2f} "
              f"crisis R={m['crisis_recall']:.2f} P={m['crisis_prec']:.2f}")
    del Xg; torch.cuda.empty_cache()
    keys = ("WA", "UA", "WF1", "CCC_v", "CCC_a", "crisis_recall", "crisis_prec")
    arr = {k: np.array([f[k] for f in results[name]]) for k in keys}
    print(f"[{name}] MEAN " + " ".join(f"{k}={arr[k].mean():.2f}±{arr[k].std():.2f}" for k in keys))


## 5. Results + artifacts

In [ ]:
# Cell 5 — aggregate (mean +/- std over folds), save 3-head artifacts + results.
import pandas as pd
HF_ID = {"wavlm-large": "microsoft/wavlm-large", "emotion2vec": "iic/emotion2vec_base"}
KEYS = ("WA", "UA", "WF1", "CCC_v", "CCC_a", "crisis_recall", "crisis_prec")

rows, agg = [], {}
for name, folds in results.items():
    agg[name] = {}
    for k in KEYS:
        vals = np.array([f[k] for f in folds])
        agg[name][k] = (float(vals.mean()), float(vals.std()))
        rows.append({"backbone": name, "metric": k,
                     "mean": round(float(vals.mean()), 3), "std": round(float(vals.std()), 3)})
df = pd.DataFrame(rows)
df.to_csv(f"{ART}/results_voice_mtl.csv", index=False)
print(df.to_string(index=False))

print(f"\n=== Multi-task summary (hard crisis-recall floor = {RECALL_FLOOR:.2f}) ===")
for name, a in agg.items():
    print(f"[{name}] emotion WA={a['WA'][0]:.2f}  |  affect CCC v={a['CCC_v'][0]:.2f}/a={a['CCC_a'][0]:.2f}"
          f"  |  crisis recall={a['crisis_recall'][0]:.2f} prec={a['crisis_prec'][0]:.2f}")

# save a serving/test bundle per backbone (fold-0 head).
for name, head in artifacts.items():
    d = f"{ART}/artifact_{name}"
    os.makedirs(d, exist_ok=True)
    torch.save(head.state_dict(), f"{d}/mtl_head.pt")
    cfg = {"backbone_hf_id": HF_ID[name], "backbone": name, "embed_dim": BACKBONES[name]["dim"],
           "emotions": EMOTIONS, "valence_arousal": VALENCE_AROUSAL, "crisis_set": sorted(CRISIS),
           "recall_floor": RECALL_FLOOR, "crisis_tau": results[name][0]["tau"],
           "sample_rate": SR, "max_samples": MAX_SAMPLES, "frame_hop": FRAME_HOP, "t_max": T_MAX,
           "pooling": "masked-mean", "head": "mtl-3head", "head_dim": HEAD_DIM}
    with open(f"{d}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"  saved bundle -> {d}")

with open(f"{ART}/results_voice_mtl.json", "w") as f:
    json.dump({"agg": agg, "per_fold": results, "recall_floor": RECALL_FLOOR,
               "n_folds": N_FOLDS, "split": SPLIT, "base_seed": BASE_SEED}, f, indent=2)
print("\nartifacts in /kaggle/working: results_voice_mtl.{csv,json}, sample_val.wav, "
      "artifact_*/ (mtl_head.pt + config.json)")
